# Topic modelling 

This workbook implements the topic modelling steps. 

1. Load the dataset  
2. Pre-process the data using the functions in nlp_tasks.py
3. Apply BERTopic model 
4. Save the topic model
5. Categorise whether topics are material or non-material 


In step 1, I use named entity recognition (NER) - with the pre-trained HuggingFace model [reddit-ner-place-names](https://huggingface.co/cjber/reddit-ner-place_names) developed by Cillian Berragan at Liverpool uni. 

In [13]:
import pandas as pd
import numpy as np
import re
import string 
from datasets import Dataset

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster import hierarchy as sch

from transformers import AutoModelForMaskedLM, pipeline, AutoTokenizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

import nltk 
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from gensim import corpora
from gensim.models import LdaModel

from hdbscan import HDBSCAN
from umap import UMAP

# define the model 
# model = AutoModelForMaskedLM.from_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

[nltk_data] Downloading package stopwords to /Users/bea/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/bea/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/bea/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load data
df = pd.read_csv('../outputs/train_comments.csv')

In [3]:
# define the model checkpoint
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Process the data to remove place names, numbers and non-ascii characters

In [4]:
nlp = NLP_Tasks(model_checkpoint)

# remove place names, numbers and non-ASCII characters
df = nlp.remove_place_names(df=df, column='comment_text')
df = nlp.remove_numbers(df=df, column='cleaned_comment_text')
df = nlp.remove_non_ascii(df=df, column='cleaned_comment_text')

Device set to use mps:0


In [32]:
df_split

address  \
0          61 Commodore House 2 Admiralty Avenue Silvertown London E16 2PY   
1              20 Rendel Apartments 3 Lock Side Way Beckton London E16 2HA   
2              20 Rendel Apartments 3 Lock Side Way Beckton London E16 2HA   
3              20 Rendel Apartments 3 Lock Side Way Beckton London E16 2HA   
4              20 Rendel Apartments 3 Lock Side Way Beckton London E16 2HA   
...                                                                    ...   
2490                       28 Sheerness Mews North Woolwich London E16 2SR   
2491                     104 Sheldrake Close North Woolwich London E16 2HT   
2492  212 John Cabot House 40 Royal Crest Avenue Silvertown London E16 2BA   
2493  212 John Cabot House 40 Royal Crest Avenue Silvertown London E16 2BA   
2494                                                                   NaN   

       stance             date  \
0     Objects  Sun 20 Jun 2021   
1     Objects  Fri 21 Feb 2020   
2     Objects  Fri 21 Feb 2020   
3     Objects  Fri 21 Feb 2020   
4     Objects  Fri 21 Feb 2020   
...       ...              ...   
2490  Objects  Wed 12 Feb 2020   
2491  Objects  Sat 15 Feb 2020   
2492  Objects  Thu 10 Jun 2021   
2493  Objects  Thu 10 Jun 2021   
2494  Objects  Tue 31 Aug 2021   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

Splits the text on newlines 

In [5]:
# split text on newlines
df_split = nlp.split_text_on_newline(df=df, column='cleaned_comment_text')

In [6]:
cleaned_place_text = df_split['cleaned_comment_text'].tolist()

Seed topics 

In [68]:
seed_topic_list = [
    ["large-scale", "expansion", "size of development", "massive project", "extent", "scale"],
    ["populaiton", "density", "overcrowding", "high-rise", "too many units", "population increase", "compact"],
    ["loss of light", "block sunglight", "sunlight", "darkness", "shadow", "sunlight obstruction"],
    ["loss of privacy", "privacy", "overlooked", "overlook", "overlooking", "see in"],
    ["loss of view", "view", "obstructed view", "view obstruction", "view blocked", "view blocked", "window"],
    ["noise", "disturbance", "loud", "loud noise", "noise pollution", "sound", "hear"],
    ["traffic congestion", "traffic", "road congestion", "road blockage", "traffic jam", "road block", "busy road", "congestion", "rush hour"],
    ["parking problems", "parking issues", "parking space", "parking lot", "parking availability", "car space"],
    ["construction traffic", "construction vehicles", "construction site traffic", "work site traffic"],
    ["dust and debris", "dirt", "dust", "debris", "construction dust", "construction debris"],
    ["wildlife", "habitat", "wildlife habitat", "animal habitat", "animal life", "wildlife protection", "squirrel", "bird", "animal", "nature"]
]

max_len = max([len(topic) for topic in seed_topic_list])
padded_list = [topic + [""]*(max_len-len(topic)) for topic in seed_topic_list]

Set the parameters of the BERTopic model 

In [69]:
# The UMAP model is used to reduce the dimensionality of the data
# note: random_state controls the seed - allowing for reproducible maps 
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=43)

# The HDBSCAN model is used to cluster the data
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [70]:
topic_model = BERTopic(embedding_model=model, hdbscan_model=hdbscan_model, umap_model=umap_model, verbose=True, calculate_probabilities=True, seed_topic_list=padded_list)

topics, probs = topic_model.fit_transform(cleaned_place_text)

2025-04-09 15:23:33,145 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 78/78 [00:24<00:00,  3.17it/s]
2025-04-09 15:24:00,440 - BERTopic - Embedding - Completed ✓
2025-04-09 15:24:00,442 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:02<00:00,  2.29s/it]
2025-04-09 15:24:02,790 - BERTopic - Guided - Completed ✓
2025-04-09 15:24:02,791 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-09 15:24:16,303 - BERTopic - Dimensionality - Completed ✓
2025-04-09 15:24:16,304 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-09 15:24:16,663 - BERTopic - Cluster - Completed ✓
2025-04-09 15:24:16,668 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-04-09 15:24:16,791 - BERTopic - Representation - Completed ✓


In [71]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,641,-1_the_and_to_of,"[the, and, to, of, in, for, this, is, will, be]","[I have commented above that the access across the parking area is too narrow and does not meet the basic requirements for main vehicular access of the kind proposed by the development. I should also mention that as the access is over brick paving only ever intended for domestic vehicles that it will be wholly inadequate to support construction traffic or other heavy vehicles. The existing Development will have to pay for the maintenance of the access across the Car Park, while deriving no benefit from it, and as its access over private parking area will be expected to insure it. There is no provision in the reservation of rights which the developers are relying contained in the Transfer of the land dated th October to make any charges for the access. When this was originally the case it was only occasional access by the Management Authority and in fact they maintain a very large security gate in place at the moment as the number of vehicle movements are so low. Once there are more than a dozen three bedroom houses the number of vehicle movements e.g. delivery vehicles, waist collections etc. and HGV, construction equipment etc. during the construction phase will undoubtedly greatly increase. I understand encouragement for people to use cycles are planned again concerns of safety if ""cycle lanes"" are not in place., ) It is planned to access the development through and . This will have a substantial impact for residents during construction and will increase traffic flow subsequently as it will be the main access for the proposed new buildings. This will create noise pollution, congestion and a general loss of amenity to residents and is likely to cause additional parking issues on estate roads., I object this planning permission because along with other residents, we do not wish to be subjected to more building works. Most of us moved into the a few years ago on the proviso that the majority of works would come to an end and another construction site is not what we bargained for. No one wants further noise and pollution. Especially given that many residents are now working from home and will even continue to do so in the foreseeable future due to new hybrid working arrangements. Above all, the loss of sunlight and effects to the neighbouring buildings is being neglected. Overall the houses more than , residents and is there really a need for this block to further increase the population which already struggles with capacity at the nearby and also stock levels in the local . If more social housing is needed I suggest the Council finds alternative land or moves the claimants outside of . Also this plan was not in the original deletion Et plan which again - many of us have invested in this area and now live here because we expected the works to eventually complete - we did not expect for small parts of land to be sold off and neglect the opinions of the residents who have effectively been ""mis-sold"" what the area would look like if the planning permission is granted.]"
1,0,198,0_green_space_was_this,"[green, space, was, this, would, the, of, as, in, development]","[Owners at Royal Wharf were sold on the significant green space that would form part of the development. Approved planning applications for the development previously emphasised the amount of green space. Ballymore now intend to take this away, having misled leaseholders into believing it would remain green space., Loss of amenity. Owners at were sold on the significant green space that would form part of the development. Approved planning applications for the development previously emphasised the amount of green space. now intend to take this away, having misled leaseholders into believing it would remain green space., Loss of amenity. Owners at were sold on the significant green space that would form part of the development. Approved pla

Identify topics for a specific application

In [72]:
cs = CommentsSaver(env='prod')
df = cs.read_all()
df_ealing = df[df['application_id']=='223203FUL']

Connecting to the ai4ci-db database...
Successfully connected to ai4ci-db.


In [73]:
# remove place names, numbers and non-ASCII characters
df_ealing = nlp.remove_place_names(df=df_ealing, column='comment_text')
df_ealing = nlp.remove_numbers(df=df_ealing, column='cleaned_comment_text')
df_ealing = nlp.remove_non_ascii(df=df_ealing, column='cleaned_comment_text')

# split text on newlines
df_ealing_split = nlp.split_text_on_newline(df=df_ealing, column='cleaned_comment_text')

In [74]:
cleaned_ealing_text = df_ealing_split['cleaned_comment_text'].tolist()

In [75]:
# Apply previous topic model to the new dataset of comments from the one application in Ealing 
topics_subset, probs_subset = topic_model.transform(cleaned_ealing_text)

Batches: 100%|██████████| 16/16 [00:04<00:00,  3.32it/s]
2025-04-09 15:24:49,824 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-04-09 15:24:52,960 - BERTopic - Dimensionality - Completed ✓
2025-04-09 15:24:52,961 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-04-09 15:24:52,984 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-04-09 15:24:53,182 - BERTopic - Probabilities - Completed ✓
2025-04-09 15:24:53,183 - BERTopic - Cluster - Completed ✓


In [76]:
probs_array = np.array(probs_subset)

In [77]:
probs_array[1]>0.05

array([False, False, False,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False])

In [78]:
# Get the boolean mask for probs_array[1] > 0.05
mask = probs_array[1] > 0.04

# Filter the topics based on the mask
topic_info = topic_model.get_topic_info()[1:]
filtered_topics = topic_info[mask]

# Print the filtered topics
print(filtered_topics)

   Topic  Count                       Name  \
4      3    100  3_height_high_street_area   

                                                          Representation  \
4  [height, high, street, area, the, keeping, design, of, is, buildings]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [79]:
probs_subset

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.06057606e-02, 6.87393629e-03, 8.42254344e-03, ...,
        4.76558157e-03, 6.78553653e-03, 1.08342750e-02],
       [2.76806164e-03, 2.36185529e-03, 2.71675607e-03, ...,
        2.20338719e-03, 2.43480284e-03, 4.35427413e-03],
       ...,
       [3.16551926e-03, 4.17906731e-03, 3.09003216e-03, ...,
        8.99475615e-04, 2.34413942e-02, 3.39561072e-03],
       [7.84846409e-01, 5.31992492e-04, 1.18453322e-03, ...,
        3.11830687e-04, 4.36665727e-04, 1.01263305e-03],
       [1.11872995e-08, 1.37950393e-08, 1.09295594e-08, ...,
        1.02773832e-08, 6.55351138e-03, 1.21592165e-08]])

In [80]:
cleaned_ealing_text[1]

'Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area, and only add more pressure to the already over-stretched infrastructure and services of . On top of that, the electrical power requirements of these developments will severely strain our services to breaking point. Fully object.'

In [81]:
df_ealing

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,cleaned_comment_text
13038,9457,Ealing,223203FUL_112,223203FUL,34 lexden road london w3 9nz London W3 9NZ W3 9NZ,Objects,2022-08-25,Taking away trees .enough of highrise flats in the area.we need' privacy.Please consider local residence New '..Road leading to Lexden rd not acceptable.To much traffic.,2025-03-21,Taking away trees .enough of highrise flats in the area.we need' privacy.Please consider local residence New '..Road leading to not acceptable.To much traffic.
13048,9355,Ealing,223203FUL_21,223203FUL,10 Baldwyn Gardens London W3 6HH W3 6HH,Objects,2022-09-16,"Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area, and only add more pressure to the already over-stretched infrastructure and services of Acton. On top of that, the electrical power requirements of these developments will severely strain our services to breaking point. Fully object.",2025-03-21,"Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area, and only add more pressure to the already over-stretched infrastructure and services of . On top of that, the electrical power requirements of these developments will severely strain our services to breaking point. Fully object."
13102,9332,Ealing,223203FUL_16,223203FUL,35 Cumberland Park London w3 6sx w3 6sx,Objects,2022-09-16,"The proposals will not only be an eyesore and increase traffic significantly on the already excessively busy Horn Lane, they are out of keeping with the general architecture of the area. The new tower blocks nearby are already a blot on the landscape; these proposals would only add to that",2025-03-21,"The proposals will not only be an eyesore and increase traffic significantly on the already excessively busy , they are out of keeping with the general architecture of the area. The new tower blocks nearby are already a blot on the landscape; these proposals would only add to that"
13105,9371,Ealing,223203FUL_35,223203FUL,34 Derwentwater Road LONDON W3 6DF W3 6DF,Objects,2022-09-10,The centre of Acton is overcrowded. The new block over Ark Soane school is already introduced a huge high building into the area. Also facilities like doctors surgeries are impossibly busy. How will they cope with hundreds more people in the area?,2025-03-21,The centre of is overcrowded. The new block over is already introduced a huge high building into the area. Also facilities like doctors surgeries are impossibly busy. How will they cope with hundreds more people in the area?
13107,9336,Ealing,223203FUL_1,223203FUL,177 Horn Lane Flat 2 London W3 6PW W3 6PW,Neutral,2022-10-09,"In the shrubs next to the road you can find shrews, voles or something like this. Has a survey been done , what were the results and how do you plan to recreate this habitat you are destroying? Thanks",2025-03-21,"In the shrubs next to the road you can find shrews, voles or something like this. Has a survey been done , what were the results and how do you plan to recreate this habitat you are destroying? Thanks"
...,...,...,...,...,...,...,...,...,...,...
13274,9508,Ealing,223203FUL_156,223203FUL,6 Tolkien House London W3 8YJ W3 8YJ,Objects,2022-08-15,"Too much pressure on an already extremely populated area.\nThere are not enough local services to support another development in Acton. Traffic is already horrible as it is, no need for more buildings. It is hard to find a gp and schools are oversubscribed.",2025-03-21,"Too much pressure on an already extremely populated area.\nThere are not enough local services to support another development in . Traffic is already horrible as it is, no need for more buildings. It is hard to find a gp and schools are oversubscribed."
13275,9509,Ealing,223203FUL_157,223203FUL,61 Emanuel Ave Acton London W3 6JQ W3 6JQ,Objects,2

In [82]:
# Identify topics with a probability > 0.5 in at least one entry
selected_topics = np.where(probs_array > 0.5)[1]  
selected_topics = np.unique(selected_topics)  

# Find the main topic for each document
main_topics = np.argmax(probs_array, axis=1)

# Count occurrences of each main topic
topic_counts = pd.Series(main_topics).value_counts().reset_index()
topic_counts.columns = ["Topic", "Count"]

# Get topic representations from BERTopic (full version)
topic_info = topic_model.get_topic_info()
filtered_topics = topic_info[topic_info["Topic"].isin(selected_topics)][["Topic", "Representation"]]

# Merge with topic counts
final_topics = filtered_topics.merge(topic_counts, on="Topic", how="left").fillna(0)

# Get representative documents: Take 3 example documents for each topic
representative_docs = {}
for topic in selected_topics:
    doc_indices = np.where(main_topics == topic)[0]  # Get indices of docs assigned to this topic
    top_docs = [cleaned_ealing_text[i] for i in doc_indices[:3]]  # Get up to 3 docs
    representative_docs[topic] = top_docs

# Add representative documents to the dataframe
final_topics["Representative Docs"] = final_topics["Topic"].map(representative_docs)

# Display full data
pd.set_option("display.max_colwidth", None)  # Ensure full text is shown
final_topics

,Topic,Representation,Count,Representative Docs
0,0,"[green, space, was, this, would, the, of, as, in, development]",34,"[. Removal of mature trees and green space, Removal of mature trees and green space, What about electricity we have no capacity on the grid in .]"
1,2,"[school, nursery, noise, next, dust, children, primary, construction, building, and]",25,"[. Open space - the proposed increase in in residential properties, with resulting lack of open space is also of particular concern. In a post COVID world, planners should be creating developments with more open space not less., . Open space - the proposed increase in in residential properties, with resulting lack of open space is also of particular concern. In a post COVID world, planners should be creating developments with more open space not less., I take my children swimming and to meet friends in , from south, and am disappointed to see more mature trees being removed to make space for yet another high rise building, which is out of keeping with local surroundings. Please reconsider this proposal. Thank you.]"
2,3,"[height, high, street, area, the, keeping, design, of, is, buildings]",53,"[Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area, and only add more pressure to the already over-stretched infrastructure and services of . On top of that, the electrical power requirements of these developments will severely strain our services to breaking point. Fully object., . Over development and out of keeping - the plans as presented are completely out of keeping with the local area. While and are storeys, they are an anomaly in . While some sensitive development of the site and improvements for existing residents is surely welcome, buildings of this height and mass will dominate the local skyline, reduce light and amenity for neighbours and put a strain on existing local facilities. The council should be looking to build to a much lower height to better integrate the site with the surrounding area., . Over development and out of keeping - the plans as presented are completely out of keeping with the local area. While and are storeys, they are an anomaly in . While some sensitive development of the site and improvements for existing residents is surely welcome, buildings of this height and mass will dominate the local skyline, reduce light and amenity for neighbours and put a strain on existing local facilities. The council should be looking to build to a much lower height to better integrate the site with the surrounding area.]"
3,4,"[dlr, already, services, amenities, bus, service, tfl, stretched, gym, with]",29,"[The centre of is overcrowded. The new block over is already introduced a huge high building into the area. Also facilities like doctors surgeries are impossibly busy. How will they cope with hundreds more people in the area?, This area is becoming massively over developed. Traffic is terrible around already and the doctors/local services are over populated too. It seems any tiny space left anywhere suddenly has to have a great tower of flats on it., Overdevelopment without increase in required facilities e.g. capacity in local schools, doctors surgeries, hospitals etc]"
4,6,"[pollution, noise, polluted, airport, area, air, traffic, distribution, the, of]",23,"[Environmental impact;, The proximity to neighbouring buildings will eliminate all natural sunlight into the neighbouring properties. This will have an impact on living standards and resident mental health. Mature trees will be removed against the current environment commitments and will particularly affect residents of . The buildings are too tall, too close to the neighbouring ones and the destatiction of natural assets is completely against the unitary develop plan., Lexden Road is currently a quiet side road away from the pollution of the main road. The air quality assessment finding that the dev